In [1]:
import datetime

from app.services.db_service import DbService
from app.services.external_api.searoute_api import SearouteApi

from dotenv import load_dotenv

load_dotenv()

sql_db_service = DbService()
await sql_db_service.init_pool()
searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")


In [12]:
route, err = await sql_db_service.get_route_by_id("f5a102f0-5769-4e85-a947-fdb0199a67e8")

In [2]:
route, err = await sql_db_service.get_route_by_id("f5a102f0-5769-4e85-a947-fdb0199a67e8")
#departure_port, err = await sql_db_service.get_port_by_id(route.departure_port_id)
#destination_port, err = await sql_db_service.get_port_by_id(route.destination_port_id)
departure_port, err = await sql_db_service.get_port_by_locode("RULED")
destination_port, err = await sql_db_service.get_port_by_locode("AEJEA")

sea_route, err = await searoute_api.build_sea_route(departure_port.latitude, departure_port.longitude, destination_port.latitude, destination_port.longitude, is_plan=True, speed_in_knots=10, departure_dt=datetime.datetime.now())

In [3]:
err

In [4]:
departure_port

SeaPortDB(locode='RULED', country_code='', country_name='Russian Federation', port_name='Saint Petersburg (ex Leningrad)', latitude=59.916139, longitude=30.241585, rank_score=None, similarity_score=None, combined_score=None, match_type='unknown', mabux_ids=[456, 136, 11, 124, 281, 397], port_size='large', mabux_id=11, barge_status=None, truck_status=None, agent_contact_list=None, bubble_id='1713117931652x665026778720616600', search_key='saint petersburg (ex leningrad)|ruled|russian federation|11', id='76214809-b928-41ce-bca1-dbc89f9e2178')

In [5]:
destination_port

SeaPortDB(locode='AEJEA', country_code='', country_name='United Arab Emirates', port_name='Jebel Ali Free Zone', latitude=25.017737, longitude=55.010241, rank_score=None, similarity_score=None, combined_score=None, match_type='unknown', mabux_ids=[283, 139, 3, 318], port_size='large', mabux_id=318, barge_status=True, truck_status=True, agent_contact_list=None, bubble_id='1713117933687x933559115491074300', search_key='jebel ali free zone|aejea|united arab emirates|318', id='977d403e-e7e2-483d-a4ff-5b40fd4211b2')

In [6]:
sea_route

SearoutePath(distance=13843309.0, departure=1767791856706, arrival=1770487570706, duration=2695714000.0, routeAreas=[SearouteArea(id=11174, name='East Oland', latitude=56.362907400048, longitude=18.75084670515933), SearouteArea(id=21542, name='Strait of Hormuz JWC', latitude=25.759466656261125, longitude=56.497759771737385), SearouteArea(id=11117, name='Suez Canal', latitude=30.011419989000046, longitude=32.57154381600009), SearouteArea(id=21615, name='Yemen Coast JWC', latitude=13.302174381549104, longitude=47.414293661101574), SearouteArea(id=21616, name='Red Sea', latitude=20.07047547245444, longitude=38.68727526262015), SearouteArea(id=21650, name='High Risk Area', latitude=5.624733314278396, longitude=48.720140579192304), SearouteArea(id=21651, name='North Bornholm', latitude=55.354520660054924, longitude=14.658282016858777), SearouteArea(id=11158, name='Gela Canal', latitude=37.20322544177797, longitude=11.91950840824596), SearouteArea(id=11100, name='Gibraltar', latitude=35.9505

In [7]:
import asyncio

async def get_ports_with_prices_every_5th_point_sql(
    sea_route,
    radius_km: float = 500.0,
    limit: int = 50,
):
    seen = {}
    coords = sea_route.seaRouteCoordinates

    for idx in range(0, len(coords), 5):
        c = coords[idx]

        ports, err = await sql_db_service.search_ports_nearby_with_prices(
            c.latitude,
            c.longitude,
            n=limit,
            radius_km=radius_km,
        )
        if err or not ports:
            continue

        for p in ports:
            existing = seen.get(p.locode)
            if not existing:
                seen[p.locode] = p

        await asyncio.sleep(0)

    result = list(seen.values())
    result.sort(key=lambda p: p._distance_km)
    return result

In [8]:
ports = await get_ports_with_prices_every_5th_point_sql(sea_route)

In [9]:
unique_ports = []
locodes = set()
for p in ports:
    if p.locode not in locodes:
        locodes.add(p.locode)
        unique_ports.append(p)


In [10]:
len(unique_ports)

1565

In [13]:
from app.data.dto.main.SeaPort import SeaPortDB
from app.data.dto.searoute.SearoutePort import SearoutePort
from typing import Tuple, List, Optional
from app.data.dto.main.BunkeringStep import BunkeringStep

def _adjust_from_weekend(date: datetime.date) -> datetime.date:
    # 5 = Saturday, 6 = Sunday
    while date.weekday() >= 5:
        date -= datetime.timedelta(days=1)
    return date

async def _find_fuel_price(
        port: SeaPortDB,
        fuel_name: str,
        date: datetime.datetime.date,
) -> Optional[float]:
    """
    Tries to find fuel price by:
    1) port locode
    2) alternative mabux ids
    Stops immediately when a price is found.
    """
    date = _adjust_from_weekend(date)

    price_db, err = await sql_db_service.get_port_fuel_price_by_port_locode(
        port.locode, fuel_name, date
    )
    if price_db:
        return price_db.value

    alt_ids, err = await sql_db_service.get_alternative_mabux_ids(port.locode.strip())
    if err or not alt_ids:
        return None

    for mabux_id in alt_ids:
        price_db, err = await sql_db_service.get_port_fuel_price_by_port_mabux_id(
            mabux_id, fuel_name, date
        )
        if price_db:
            if price_db.value > 0:
                return price_db.value

    return None

async def build_bunkering_step_dict(
    port_db: SeaPortDB,
    fuels: list,
    price_date: datetime.date,
    semaphore: asyncio.Semaphore,
) -> Optional[dict]:
    async with semaphore:
        fuel_tasks = {
            fuel.name: _find_fuel_price(
                port=port_db,
                fuel_name=fuel.name,
                date=price_date,
            )
            for fuel in fuels
        }

        prices = await asyncio.gather(*fuel_tasks.values())

        fuel_info = {}
        prices_count = 0
        prices_sum = 0.0

        for fuel_name, price in zip(fuel_tasks.keys(), prices):
            if price is not None:
                prices_count += 1
                prices_sum += price

            fuel_info[fuel_name] = {
                "mobux_price_status": price is not None,
                "fuel_name": fuel_name,
                "fuel_price": price,
                "quantity": None,
            }

        if prices_count == 0:
            return None

        return {
            "port": port_db,
            "_distance_km": getattr(port_db, "_distance_km", None),
            "fuel_info": fuel_info,
            "prices_count": prices_count,
            "prices_sum": prices_sum,
        }

async def build_bunkering_steps_dict(
    ports: List[SeaPortDB],
    fuels: list,
    price_date: datetime.date,
    max_concurrency: int = 10,
) -> List[dict]:
    semaphore = asyncio.Semaphore(max_concurrency)

    tasks = [
        build_bunkering_step_dict(
            port_db=p,
            fuels=fuels,
            price_date=price_date,
            semaphore=semaphore,
        )
        for p in ports
      #  if p.port_size == "large"
    ]

    results = await asyncio.gather(*tasks)

    steps = [r for r in results if r is not None]

    # sort by detour distance only (route info comes later)
    steps.sort(key=lambda s: s["_distance_km"] or float("inf"))

    return steps

bunkering_steps = await build_bunkering_steps_dict(
            ports = unique_ports,
            fuels=route.fuels,
            price_date=datetime.date.today()
        )

In [21]:
priced_ports = [s['port'] for s in bunkering_steps if s['prices_count'] > 0]
len(priced_ports)

1168

In [2]:
priced_ports

NameError: name 'priced_ports' is not defined

In [15]:
 indexed = []

 if departure_port:
    indexed.append(departure_port.to_indexed2("UP", "blue", "medium", True))

if destination_port:
    indexed.append(destination_port.to_indexed2("DOWN", "red", "medium", True))

for i, port in enumerate(priced_ports, 1):
    indexed.append(port.to_indexed2(str(i), "gray", "medium", False))


In [16]:
len(indexed)

1170

In [17]:
from app.services.internal_api.map_builder_api import MapBuilderApi
map_image_api = MapBuilderApi("https://api.thebunkering.com", "https://api.thebunkering.com")
image, image_err = await map_image_api.render_map(sea_route.seaRouteCoordinates, indexed)
image_err

In [18]:
with open("image.png", "wb") as f:
    f.write(image)

In [1]:
import math
import datetime
from typing import List

def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))

def find_nearest_waypoint(step, waypoints):
    nearest_wp = None
    min_dist = float("inf")

    for wp in waypoints:
        d = haversine_km(
            step.get("port").latitude,
            step.get("port").longitude,
            wp.latitude,
            wp.longitude,
        )
        if d < min_dist:
            min_dist = d
            nearest_wp = wp

    return nearest_wp


def find_nearest_coord_index(step, coordinates) -> int:
    min_dist = float("inf")
    min_idx = 0

    for i, c in enumerate(coordinates):
        d = haversine_km(
            step.get("port").latitude,
            step.get("port").longitude,
            c.latitude,
            c.longitude,
        )
        if d < min_dist:
            min_dist = d
            min_idx = i

    return min_idx


def distance_from_start_to_index(coordinates, idx: int) -> float:
    total = 0.0

    for i in range(1, idx + 1):
        prev = coordinates[i - 1]
        curr = coordinates[i]
        total += haversine_km(
            prev.latitude,
            prev.longitude,
            curr.latitude,
            curr.longitude,
        )

    return total


def enrich_ports_with_eta_and_distance(
    steps: List[dict],
    sea_route,
) -> None:
    coordinates = sea_route.seaRouteCoordinates
    waypoints = sea_route.waypoints

    for step in steps:
        # --- ETA from nearest waypoint ---
        wp = find_nearest_waypoint(step, waypoints)
        step["eta_datetime"] = wp.eta_datetime if wp else None

        # --- distance along route ---
        idx = find_nearest_coord_index(step, coordinates)
        step["distance"] = distance_from_start_to_index(coordinates, idx)



KeyboardInterrupt: 

In [ ]:
enrich_ports_with_eta_and_distance(bunkering_steps, sea_route)

In [ ]:
len(bunkering_steps)